# OBLIQ-Bench Retrieval Pipeline: Posterior Row Confidence Probing
This notebook demonstrates testing the LLM verifier signal as a first-stage retriever based on the GridProbe/OBLIQ-Bench methodology, utilizing true posterior reading (logits) instead of text generation.


In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00


In [ ]:
import json
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


In [ ]:
# Load the sample dataset
with open('obliq_twitter_sample.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

queries = dataset['queries']
documents = dataset['documents']

query_text = queries[0]['text']
print(f"Query: {query_text}")
print(f"Total documents: {len(documents)}")
# Show golden docs for reference
golden_docs = [d for d in documents if d['is_golden']]
print(f"Golden docs count: {len(golden_docs)}")



Query: find tweets expressing skepticism toward science without saying so explicitly
Total documents: 50
Golden docs count: 3


In [ ]:
from transformers import BitsAndBytesConfig

In [ ]:
# Initialize Qwen Model
# You can change to "Qwen/Qwen2.5-7B-Instruct" if you have a larger GPU.
model_id = "Qwen/Qwen3-4B-Instruct-2507" # qwen 2.5 is horrible
quant_config = BitsAndBytesConfig(load_in_4bit=True)
print(f"Loading {model_id}...")
# Using 4-bit quantization to easily fit in standard Colab T4 GPU
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quant_config,
)

# We still keep the pipeline around just for generating the initial aspects
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("Model loaded successfully!")



Loading Qwen/Qwen3-4B-Instruct-2507...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
# Hybrid Aspect Generation
# We simulate the hybrid approach by asking the LLM to map the query to core structural/latent aspects.

def generate_aspects(query):
    prompt = f"""System: You are an expert search taxonomist. Break down the following oblique query into 3-5 core latent aspects.
These aspects should capture the underlying meaning, structure, or implied intent, not just keyword synonyms.
Output ONLY a JSON list of strings representing the aspects.

Query: {query}
"""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) # tokenizer = AutoTokenizer.from_pretrained(model_id)
    out = pipe(text, max_new_tokens=150, temperature=0.1, return_full_text=False)[0]['generated_text']

    try:
        # Extract JSON list
        start = out.find('[')
        end = out.rfind(']') + 1
        aspects = json.loads(out[start:end])
    except:
        # Fallback aspects if parsing fails
        aspects = [
            "Expression of doubt regarding scientific consensus",
            "Sarcastic or dismissive tone towards experts",
            "Citing changing guidelines as proof of unreliability"
        ]
    return aspects

aspects = generate_aspects(query_text)
print("Generated Aspects:")
for i, a in enumerate(aspects):
    print(f"{i+1}. {a}")



[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (i

Generated Aspects:
1. implicit criticism of scientific authority
2. subtle resistance to scientific consensus
3. use of social media to veil dissent
4. expression of doubt through indirect language
5. contextual framing of science as politically or ideologically loaded


In [ ]:
# Chunking Strategy: Positional Chunk Sampling (Rows)
# We treat the corpus as a flat sequence and group them into rows of K consecutive chunks.
K = 5 # 5 consecutive documents per row

rows = []
for i in range(0, len(documents), K):
    row_docs = documents[i:i+K]
    row_id = f"row_{i//K + 1}"
    rows.append({
        "row_id": row_id,
        "docs": row_docs,
        "contains_golden": any(d['is_golden'] for d in row_docs)
    })

print(f"Created {len(rows)} rows with up to {K} chunks each.")



Created 10 rows with up to 5 chunks each.


In [ ]:
# Query: {query}
# removed this from the prompt

In [ ]:
# Prompt for Row Probing via Posterior (Logits)
import string

def probe_row_posterior(query, aspects, row_docs): # will go over per row
    chunks_text = ""
    for idx, doc in enumerate(row_docs):
        chunks_text += f"Chunk {idx+1} [ID: {doc['doc_id']}]: {doc['text']}\n\n" # chunk id, doc id, doc text

    # Map aspects to letters A, B, C...
    letters = list(string.ascii_uppercase)
    aspect_options = []
    for i, a in enumerate(aspects):
        aspect_options.append(f"{letters[i]}) {a}") # store letters as numbers, and map them to the aspects

    none_letter = letters[len(aspects)]
    aspect_options.append(f"{none_letter}) None of the above are evident") # option 6

    options_str = "\n".join(aspect_options)

    # Simple prompt, works
    prompt = f"""You are an expert reasoning AI evaluating a set of text chunks.
DO NOT rely on direct word matching. Detect implicit signals, analogue structures, or tip-of-the-tongue recollections.


Chunks in this row:
{chunks_text}

Aspects:
{options_str}

Question: Of the aspects above, which is most evident in these chunks? Even if only ONE chunk is relevant, choose the corresponding aspect. If absolutely none are relevant, choose {none_letter}.
Answer with a single letter (e.g. A, B, C...)."""

    messages = [
        {"role": "system", "content": "You are a highly precise classifier. You only output a single letter."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) # tokenizer = AutoTokenizer.from_pretrained(model_id)
    # formats the weird message format into a string format
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Run the model (no generation, just 1 forward pass), emphasis on one forward pass :)
    with torch.no_grad():
        outputs = model(**inputs)

    # get that last logit
    next_token_logits = outputs.logits[0, -1, :]

    # Extract the token IDs for the valid options (A, B, C, etc.)
    option_letters = letters[:len(aspects) + 1]

    # Qwen tokenizer encodes single letters cleanly. We get the token IDs for these options.
    # Note: Sometimes tokenizers add a space prefix or different formats, but for standard single letters: # new info
    token_ids = [tokenizer.encode(letter, add_special_tokens=False)[0] for letter in option_letters]

    # Get the specific logits for these options
    option_logits = next_token_logits[token_ids] # relates back to the earlier line

    # Apply Softmax to get a normalized probability distribution strictly across these options
    # now is this needed?? yes
    option_probs = torch.softmax(option_logits, dim=0)

    aspect_scores = {}
    for i, aspect in enumerate(aspects):
        aspect_scores[aspect] = float(option_probs[i]) * 100 # Score from 0-100

    none_prob = float(option_probs[-1]) * 100 # ehhhhhhhhhhhhhhhhhhh

    # The paper mentions "posterior peak over aspects".
    # We take the maximum probability among the valid aspects as the overall row confidence.
    max_aspect_prob = max(aspect_scores.values())

    return {
        "aspect_scores": aspect_scores,
        "none_prob": none_prob,
        "overall_row_confidence": max_aspect_prob
    }



In [ ]:
# Run the Probing Pipeline
results = []
print("Starting Row Probing with Posteriors...")
for row in rows:
    print(f"Probing {row['row_id']}... (Contains Golden: {row['contains_golden']})")
    probe_result = probe_row_posterior(query_text, aspects, row['docs']) # pass in row['docs'], which we will enumerate in the function

    # Record results
    result_entry = {
        "row_id": row['row_id'],
        "contains_golden": row['contains_golden'],
        "golden_doc_ids": [d['doc_id'] for d in row['docs'] if d['is_golden']],
        "overall_row_confidence": probe_result['overall_row_confidence'],
        "none_probability": probe_result['none_prob'],
        "aspect_scores": probe_result['aspect_scores']
    }
    results.append(result_entry)
print("Probing completed!")



Starting Row Probing with Posteriors...
Probing row_1... (Contains Golden: False)
Probing row_2... (Contains Golden: False)
Probing row_3... (Contains Golden: True)
Probing row_4... (Contains Golden: False)
Probing row_5... (Contains Golden: False)
Probing row_6... (Contains Golden: False)
Probing row_7... (Contains Golden: False)
Probing row_8... (Contains Golden: True)
Probing row_9... (Contains Golden: True)
Probing row_10... (Contains Golden: False)
Probing completed!


In [ ]:
# Ranking and Evaluation
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='overall_row_confidence', ascending=False).reset_index(drop=True)

print("=== Ranking Results (Posterior Method) ===")
# Display formatted scores for readability
df_display = df_results.copy()
df_display['overall_row_confidence'] = df_display['overall_row_confidence'].round(2)
df_display['none_probability'] = df_display['none_probability'].round(2)
display(df_display[['row_id', 'overall_row_confidence', 'none_probability', 'contains_golden', 'golden_doc_ids']])

# Check if golden documents were retrieved successfully
golden_rows = df_results[df_results['contains_golden'] == True]
print("\n=== Golden Rows Retrieval Status ===")
for idx, row in golden_rows.iterrows():
    rank = idx + 1
    # Simple evaluation: if the golden row is ranked in the top K (where K = number of golden rows + 1), it's a success
    status = "SUCCESS" if rank <= (len(golden_rows) + 1) else "SUBOPTIMAL"
    print(f"Rank: {rank} | Row ID: {row['row_id']} | Score: {row['overall_row_confidence']:.2f} | Status: {status}")



=== Ranking Results (Posterior Method) ===


,row_id,overall_row_confidence,none_probability,contains_golden,golden_doc_ids
0,row_3,100.00,0.0,True,[d1]
1,row_8,89.50,0.0,True,[d2]
2,row_9,81.25,0.0,True,[d3]
3,row_6,0.00,100.0,False,[]
4,row_7,0.00,100.0,False,[]
5,row_4,0.00,100.0,False,[]
6,row_10,0.00,100.0,False,[]
7,row_5,0.00,100.0,False,[]
8,row_1,0.00,100.0,False,[]
9,row_2,0.00,100.0,False,[]



=== Golden Rows Retrieval Status ===
Rank: 1 | Row ID: row_3 | Score: 100.00 | Status: SUCCESS
Rank: 2 | Row ID: row_8 | Score: 89.50 | Status: SUCCESS
Rank: 3 | Row ID: row_9 | Score: 81.25 | Status: SUCCESS


In [ ]:
# Export Results
export_data = {
    "query": query_text,
    "aspects_used": aspects,
    "method": "posterior_logits",
    "row_evaluations": results
}

with open("row_posterior_probing_results.json", "w", encoding="utf-8") as f:
    json.dump(export_data, f, indent=2)

print("Successfully exported posterior probabilities to 'row_posterior_probing_results.json'.")



Successfully exported posterior probabilities to 'row_posterior_probing_results.json'.
